<a href="https://colab.research.google.com/github/jeremy26/hydranets_course/blob/claude/modernize-autoware-course-edY16/Module_3_Advanced_Heads_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3: Adding New Heads to a HydraNet — Lab

In Module 2, you trained a HydraNet with **drivable area + depth** heads.

Now you'll learn how to **add new task heads** to a frozen backbone —
the core idea behind multi-task learning in autonomous driving.

Architecture classes are imported from `hydranet.py` — no need to redefine anything.


# 1 — Setup


In [ ]:
!pip install -q torchvision pillow matplotlib numpy tqdm

# Download shared architecture module
!wget -q https://raw.githubusercontent.com/jeremy26/hydranets_course/claude/modernize-autoware-course-edY16/hydranet.py -O hydranet.py

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

from hydranet import HydraNet, INPUT_SIZE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"Input size: {INPUT_SIZE}")


# 2 — Load Pre-Trained HydraNet + Dataset

We load the trained model from Module 2 and the BDD100K dataset (for labels).


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

CHECKPOINT_DIR = '/content/gdrive/MyDrive'

# Load model
model = HydraNet(num_drivable_classes=3).to(device)
model.load_state_dict(torch.load(
    os.path.join(CHECKPOINT_DIR, 'hydranet_best.pth'),
    map_location=device,
    weights_only=True,
))
model.eval()
print("Model loaded!")

# Download dataset (for labels)
# !wget -q https://hydranets-data.s3.eu-west-3.amazonaws.com/bdd_hydranet_10k.zip && unzip -q bdd_hydranet_10k.zip

DATA_ROOT = "bdd_hydranet_10k"

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


# 3 — Pre-Compute Backbone Features

We freeze the backbone and run it once over the training set.
This gives us **neck features** that new heads train on directly —
no need to re-run the backbone every epoch.

This takes a few minutes, but saves hours of head training time.


In [ ]:
img_dir = os.path.join(DATA_ROOT, 'train', 'images')
filenames = sorted(os.listdir(img_dir))

transform = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

BATCH_SIZE = 16
all_necks, all_skips = [], []

model.eval()
print(f"Pre-computing features for {len(filenames)} images...")

with torch.no_grad():
    for start in tqdm(range(0, len(filenames), BATCH_SIZE)):
        batch_files = filenames[start:start + BATCH_SIZE]
        imgs = []
        for f in batch_files:
            img = Image.open(os.path.join(img_dir, f)).convert('RGB')
            imgs.append(transform(img))
        batch = torch.stack(imgs).to(device)

        features = model.backbone(batch)
        deep = features[4]
        ctx = model.drivable_context(deep)
        neck = model.neck(ctx, features)

        all_necks.append(neck.cpu())
        all_skips.append(features[0].cpu())

neck_features = torch.cat(all_necks, dim=0)
skip_features = torch.cat(all_skips, dim=0)

print(f"\nDone! Pre-computed features:")
print(f"  Neck: {list(neck_features.shape)}  (H/4 resolution — input for new heads)")
print(f"  Skip: {list(skip_features.shape)}  (H/2 resolution — for upsampling heads)")


# 4 — Training Utilities

Generic dataset and training loop that work with any segmentation head.


In [ ]:
class PrecomputedDataset(Dataset):
    """Dataset that pairs pre-computed features with task labels."""

    def __init__(self, neck_features, skip_features, filenames,
                 label_dir, label_suffix='.png', target_size=None):
        self.neck = neck_features
        self.skip = skip_features
        self.filenames = filenames
        self.label_dir = label_dir
        self.label_suffix = label_suffix
        self.target_size = target_size

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        name = self.filenames[idx].replace('.jpg', self.label_suffix)
        label_path = os.path.join(self.label_dir, name)
        label = np.array(Image.open(label_path))

        if self.target_size:
            label = np.array(Image.fromarray(label).resize(
                (self.target_size[1], self.target_size[0]),
                Image.NEAREST))

        return {
            'neck': self.neck[idx],
            'skip': self.skip[idx],
            'label': torch.tensor(label, dtype=torch.long),
            'filename': self.filenames[idx],
        }


def train_head(head, train_loader, criterion, device, num_epochs=15, lr=1e-3):
    """Generic training loop for any head using pre-computed features."""
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    history = []

    for epoch in range(num_epochs):
        head.train()
        total_loss, n = 0, 0
        for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}', leave=False):
            neck  = batch['neck'].to(device)
            label = batch['label'].to(device)

            optimizer.zero_grad()
            pred = head(neck)
            loss = criterion(pred, label)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            n += 1

        avg = total_loss / n
        history.append(avg)
        print(f"  Epoch {epoch+1:02d}/{num_epochs} | Loss: {avg:.4f}")

    return history


# 5 — Demo: Lane Detection Head

The instructor demonstrates how to add a **Lane Detection** head.
This is the pattern you'll follow in the lab.

**Recipe:**
1. Design the head (a few conv layers)
2. Create DataLoader with pre-computed features + labels
3. Train (takes minutes!)
4. Visualize results

## Step 1: Design the Head


In [ ]:
class LanesHead(nn.Module):
    """Lane segmentation head.

    Input:  neck (B, 256, H/4, W/4)
    Output: (B, 3, H/4, W/4) — bg / left lane / right lane
    """
    def __init__(self, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(256, 128, 3, padding=1), nn.GELU(),
            nn.Conv2d(128, 64, 3, padding=1),  nn.GELU(),
            nn.Conv2d(64, num_classes, 1),
        )

    def forward(self, neck):
        return self.net(neck)

lanes_head = LanesHead(num_classes=3).to(device)
print(f"LanesHead: {sum(p.numel() for p in lanes_head.parameters()):,} parameters")
print(f"That's it — 3 conv layers. The backbone does the heavy lifting.")


## Step 2: Create DataLoader


In [ ]:
neck_h, neck_w = neck_features.shape[2], neck_features.shape[3]

lane_dataset = PrecomputedDataset(
    neck_features=neck_features,
    skip_features=skip_features,
    filenames=filenames,
    label_dir=os.path.join(DATA_ROOT, 'train', 'lanes'),
    label_suffix='.png',
    target_size=(neck_h, neck_w),
)

lane_loader = DataLoader(lane_dataset, batch_size=32, shuffle=True, num_workers=2)

# Check a sample
sample = lane_dataset[0]
print(f"Neck:  {list(sample['neck'].shape)}")
print(f"Label: {list(sample['label'].shape)}")
print(f"Label classes: {torch.unique(sample['label']).tolist()}")


## Step 3: Train the Head

Only the head trains — the backbone is frozen. This is why it's fast!


In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=255)
history = train_head(lanes_head, lane_loader, criterion, device, num_epochs=15)

plt.plot(history)
plt.title('Lane Detection Head — Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()


## Step 4: Visualize Predictions


In [ ]:
LANE_COLORS = np.array([[0, 0, 0], [255, 0, 0], [0, 0, 255]], dtype=np.uint8)

lanes_head.eval()
img_dir = os.path.join(DATA_ROOT, 'train', 'images')

fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for row in range(3):
    idx = row * 50
    sample = lane_dataset[idx]

    img = np.array(Image.open(os.path.join(img_dir, sample['filename'])).resize((neck_w, neck_h)))

    with torch.no_grad():
        pred = lanes_head(sample['neck'].unsqueeze(0).to(device))
    pred_mask = pred.argmax(1).squeeze().cpu().numpy()
    gt = sample['label'].numpy()

    axes[row, 0].imshow(img)
    axes[row, 0].set_title('Image')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(LANE_COLORS[gt])
    axes[row, 1].set_title('Ground Truth')
    axes[row, 1].axis('off')

    axes[row, 2].imshow(LANE_COLORS[pred_mask])
    axes[row, 2].set_title('Prediction')
    axes[row, 2].axis('off')

plt.suptitle('Lane Detection Head — Predictions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


# 6 — Full Pipeline: 3 Tasks, 1 Forward Pass

Now let's run the **full pipeline** end-to-end on a real image:
backbone → existing heads (drivable + depth) + our new lane head.


In [ ]:
import cv2

model.eval()
lanes_head.eval()

transform_infer = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

img_path = os.path.join(DATA_ROOT, 'train', 'images', filenames[0])
img_pil = Image.open(img_path).convert('RGB')
img_tensor = transform_infer(img_pil).unsqueeze(0).to(device)

with torch.no_grad():
    # Existing heads
    driv, dep = model(img_tensor)

    # New lane head — reuse backbone features
    features = model.backbone(img_tensor)
    ctx = model.drivable_context(features[4])
    neck_out = model.neck(ctx, features)
    lane_pred = lanes_head(neck_out)

# ── Render all outputs ────────────────────────────────────────────

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
img_np = (img_tensor[0].cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

# Drivable overlay
driv_pred = F.interpolate(driv, size=INPUT_SIZE, mode='bilinear', align_corners=False)
driv_pred = driv_pred.argmax(1).squeeze().cpu().numpy()
DRIV_COLORS = np.array([[0, 0, 0], [0, 180, 0], [0, 100, 255]], dtype=np.float32) / 255.0
driv_overlay = img_np.copy()
active = driv_pred > 0
driv_overlay[active] = 0.6 * img_np[active] + 0.4 * DRIV_COLORS[driv_pred][active]

# Depth colormap
dep_pred = F.interpolate(dep, size=INPUT_SIZE, mode='bilinear', align_corners=False)
dep_pred = dep_pred.squeeze().cpu().numpy()
depth_cm = cv2.applyColorMap((dep_pred * 255).astype(np.uint8), cv2.COLORMAP_MAGMA)
depth_cm = cv2.cvtColor(depth_cm, cv2.COLOR_BGR2RGB) / 255.0

# Lane overlay
lane_up = F.interpolate(lane_pred, size=INPUT_SIZE, mode='bilinear', align_corners=False)
lane_up = lane_up.argmax(1).squeeze().cpu().numpy()
lane_overlay = img_np.copy()
lane_active = lane_up > 0
lane_colors = np.array([[0, 0, 0], [1, 0, 0], [0, 0, 1]], dtype=np.float32)
lane_overlay[lane_active] = 0.6 * img_np[lane_active] + 0.4 * lane_colors[lane_up][lane_active]

fig, axes = plt.subplots(1, 4, figsize=(24, 5))
fig.suptitle('HydraNet — 3 Tasks, 1 Forward Pass', fontsize=14, fontweight='bold')
axes[0].imshow(img_np);       axes[0].set_title('Input');    axes[0].axis('off')
axes[1].imshow(driv_overlay); axes[1].set_title('Drivable'); axes[1].axis('off')
axes[2].imshow(depth_cm);     axes[2].set_title('Depth');    axes[2].axis('off')
axes[3].imshow(lane_overlay); axes[3].set_title('Lanes');    axes[3].axis('off')
plt.tight_layout()
plt.show()
print('3 tasks from a single backbone — this is the power of HydraNet.')


# 7 — YOUR TURN: Build Your Own Head!

Now it's your turn. Pick one of the tasks below and build a head for it.

**Remember the recipe:**
1. Design the head (a few conv layers)
2. Create a DataLoader with pre-computed features + your labels
3. Train (should take ~5 min)
4. Visualize and plug into the full model

**Your input is always:** `neck` = (B, 256, H/4, W/4) from the frozen backbone.

Choose your challenge:


## Option A: 2D Object Detection Head (CenterNet-style)

Predict object center heatmaps + bounding box offsets.

**Output:**
- Heatmap: `(B, 10, H/4, W/4)` — 10 BDD100K classes (car, truck, bus, person, etc.)
- Regression: `(B, 4, H/4, W/4)` — bbox offsets (dx, dy, w, h)

**Loss:** Focal Loss for heatmap + L1 for regression

**Labels:** Use `labels_train.json` in the dataset — it contains 2D bounding boxes.


In [ ]:
# ============================================================
# OPTION A: 2D Object Detection Head
# ============================================================
# Uncomment and complete!

# class DetectionHead(nn.Module):
#     def __init__(self, num_classes=10):
#         super().__init__()
#         self.shared = nn.Sequential(
#             nn.Conv2d(256, 128, 3, padding=1), nn.GELU(),
#             nn.Conv2d(128, 64, 3, padding=1),  nn.GELU(),
#         )
#         self.heatmap = nn.Conv2d(64, num_classes, 1)
#         self.regression = nn.Conv2d(64, 4, 1)
#
#     def forward(self, neck):
#         x = self.shared(neck)
#         return self.heatmap(x), self.regression(x)
#
# # TODO: Build a dataset that reads bounding boxes from labels_train.json
# # TODO: Render Gaussian heatmaps at object centers
# # TODO: Train with Focal Loss + L1
# # TODO: Visualize detections on images


## Option B: Steering / Trajectory Prediction

Predict the steering angle from the scene.
Classification over 61 bins (-30 to +30 degrees).

**Output:** `(B, 61)` — logits over steering angle bins

**Loss:** Cross-Entropy

**Labels:** BDD100K videos have GPS/IMU data. For simplicity, derive steering
from consecutive frame GPS coordinates in `labels_train.json`.


In [ ]:
# ============================================================
# OPTION B: Steering Prediction Head
# ============================================================
# Uncomment and complete!

# class SteeringHead(nn.Module):
#     def __init__(self, num_bins=61):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.AdaptiveAvgPool2d(1),
#             nn.Flatten(),
#             nn.Linear(256, 128), nn.GELU(),
#             nn.Dropout(0.3),
#             nn.Linear(128, num_bins),
#         )
#
#     def forward(self, neck):
#         return self.net(neck)
#
# # TODO: Extract steering labels from labels_train.json GPS data
# # TODO: Discretize into 61 bins
# # TODO: Train with CrossEntropyLoss
# # TODO: Visualize: show image + predicted vs actual steering angle


## Option C: Scene Classification (Weather + Time of Day)

Predict scene attributes from BDD100K metadata.

**Output:** `(B, num_classes)` — weather (6 classes) or time of day (3 classes)

**Loss:** Cross-Entropy

**Labels:** `labels_train.json` contains `weather` and `timeofday` fields for every image.


In [ ]:
# ============================================================
# OPTION C: Scene Classification Head
# ============================================================
# Uncomment and complete!

# class SceneClassificationHead(nn.Module):
#     def __init__(self, num_classes=6):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.AdaptiveAvgPool2d(1),
#             nn.Flatten(),
#             nn.Linear(256, 128), nn.GELU(),
#             nn.Dropout(0.3),
#             nn.Linear(128, num_classes),
#         )
#
#     def forward(self, neck):
#         return self.net(neck)
#
# # Weather classes: clear, overcast, partly cloudy, rainy, snowy, foggy
# # Time classes: daytime, night, dawn/dusk
# # TODO: Parse labels_train.json for weather/timeofday labels
# # TODO: Create a PrecomputedDataset variant for classification
# # TODO: Train with CrossEntropyLoss
# # TODO: Visualize: show image grid with predicted vs actual labels


## Option D: Build Your Own Head!

Design a head for any task you can think of. Some ideas:
- **Free-space estimation** — binary mask of drivable ground
- **Traffic sign detection** — classify signs in a region
- **Road surface quality** — smooth, cracked, wet

**Template below:**


In [ ]:
# ============================================================
# OPTION D: Your Own Head
# ============================================================

# class MyHead(nn.Module):
#     def __init__(self):
#         super().__init__()
#         # Input: neck (B, 256, H/4, W/4)
#         # Output: whatever your task needs!
#         pass
#
#     def forward(self, neck):
#         pass
#
# # TODO: Define your labels and dataset
# # TODO: Train
# # TODO: Visualize
# # TODO: Plug into the full HydraNet


# Summary

In this module, you learned how to:

1. **Import a shared architecture** — `from hydranet import HydraNet`
2. **Pre-compute backbone features** — freeze the backbone, extract neck features once
3. **Design lightweight heads** — a few conv layers is all you need
4. **Train heads in minutes** — using pre-computed features
5. **Plug new heads into a live HydraNet** — single forward pass, multiple tasks

**Key takeaway:** Adding a new task is cheap — the backbone already understands
the scene. You just teach it a new output format.
